# Modul 2 — Representasi dan Karakteristik Citra Digital

**Praktikum Pengolahan Citra dan Visi Komputer (PCVK) — Minggu 2**
Jurusan Teknologi Informasi, Politeknik Negeri Malang

Notebook ini menjalankan seluruh isi **Jobsheet Minggu 2** pada lingkungan **Kaggle**
(pengganti Google Colab). Perbedaan utama dari instruksi modul:

* Tidak ada `drive.mount()` / `files.upload()`. Citra dibaca dari dataset yang sudah
  terpasang di `/kaggle/input/pcvk-images/week2` (di lokal: `datasets/week2`), diakses
  lewat variabel `BASE` dan `WEEK`.
* Tidak ada GUI, sehingga `cv.imshow()` tidak dipakai. Penampilan citra memakai
  `cv2_imshow()` (shim Colab) dan `plt.imshow()`.

Cakupan: Praktikum D1–D2 (membaca & menampilkan citra, konversi ruang warna,
keabuan, *resizing* + interpolasi, *flipping*, penggambaran geometri/teks pada kanvas
`int16` dan `int32`, `reshape`, `polylines`, menyimpan berkas), Pertanyaan Praktikum 1–5,
Tugas Penerapan 1–3, Tugas Analisa, serta Tugas Aplikasi Studi Kasus 1 & 2 (adaptif).

In [ ]:
# ==== KONFIGURASI LINGKUNGAN (jalan di Kaggle maupun lokal) ====
import os, sys, glob, math
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

def _find_base():
    for p in ('/kaggle/input/pcvk-images',
              '/kaggle/input/datasets/singgihwahyupermana/pcvk-images'):
        if os.path.isdir(p):
            return p
    _hit = glob.glob('/kaggle/input/**/pcvk-images', recursive=True)
    if _hit:
        return _hit[0]
    d = os.getcwd()
    for _ in range(6):
        cand = os.path.join(d, 'datasets')
        if os.path.isdir(cand):
            return cand
        d = os.path.dirname(d)
    return None

BASE = _find_base()
WEEK = os.path.join(BASE, 'week2')   # <-- ganti angka minggu sesuai notebook

def show(img, title='', cmap=None, figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

# Shim cv2_imshow (Colab) -> Kaggle
try:
    from google.colab.patches import cv2_imshow
except Exception:
    def cv2_imshow(img):
        show(img)

print('BASE :', BASE)
print('WEEK :', WEEK)
print('files:', sorted(os.listdir(WEEK)) if WEEK and os.path.isdir(WEEK) else 'MISSING')

## Helper tambahan

Sel header `show()` mengasumsikan citra berurutan **BGR** (hasil `cv.imread`) dan otomatis
mengonversinya ke RGB. Untuk citra yang **sudah** berurutan RGB (hasil `skimage.io.imread`
atau keluaran `cv.cvtColor`), dipakai helper `show_rgb()` berikut agar tidak dikonversi dua kali.

In [ ]:
def show_rgb(img, title='', cmap=None, figsize=(6, 4)):
    """Menampilkan citra yang SUDAH berurutan RGB (tanpa konversi tambahan)."""
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(img)
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

def show_any(img, title='', figsize=(6, 4)):
    """Tampilkan kanvas bertipe apa pun (mis. int16/int32) untuk keperluan tampilan.

    cv.cvtColor() hanya mendukung citra 8-bit/16-bit tertentu, sehingga kanvas
    int16/int32 perlu dipetakan dulu ke uint8 (nilai 0..255 tampil identik).
    """
    disp = img if img.dtype == np.uint8 else np.clip(img, 0, 255).astype(np.uint8)
    show(disp, title, figsize=figsize)

## D1 — Membaca citra (tanpa Google Drive)

Di modul, langkah 1 adalah menghubungkan Colab ke Google Drive. Di Kaggle langkah itu
digantikan dengan membaca berkas langsung dari dataset yang sudah terpasang. `BASE`
menunjuk folder dataset dan `WEEK` menunjuk subfolder minggu ini.

Tiga citra uji minggu 2: `foto_wajah.jpg` (foto aktivitas menampilkan wajah),
`potret.jpg` (orientasi potret), dan `lanskap.jpg` (orientasi lanskap).

In [ ]:
FILES = ['foto_wajah.jpg', 'potret.jpg', 'lanskap.jpg']
paths = {n: os.path.join(WEEK, n) for n in FILES}

imgs = {}
for n, p in paths.items():
    im = cv.imread(p)
    imgs[n] = im
    print(f'{n:16s} shape={im.shape}  dtype={im.dtype}  nbytes={im.nbytes}')

img = imgs['foto_wajah.jpg']   # citra utama untuk praktikum berikutnya
print('\nimg (foto_wajah.jpg) ->', img.shape, img.dtype)

## D2.1 — Menampilkan citra: `cv2_imshow()` vs `plt.imshow()`

`cv.imread()` menyimpan kanal dalam urutan **BGR**. `cv2_imshow()` (shim Colab)
mengasumsikan masukan BGR sehingga warna tampil benar, sedangkan `plt.imshow()`
mengasumsikan masukan **RGB** sehingga citra yang sama tampak dengan kanal merah–biru
tertukar. Perhatikan gambar berikut: pada `plt.imshow()` tanpa konversi, area kulit
tampak kebiruan.

In [ ]:
# (a) cv2_imshow -> menganggap masukan BGR, warna benar
cv2_imshow(img)

# (b) plt.imshow tanpa konversi -> menganggap masukan RGB, warna tertukar
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title('plt.imshow(img) TANPA konversi (kanal tertukar)')
plt.axis('off')
plt.show()

### Konversi BGR → RGB

Dua cara yang setara: `cv.cvtColor(img, cv.COLOR_BGR2RGB)` atau membalik urutan kanal
dengan *slicing* `img[:, :, ::-1]`.

In [ ]:
img_rgb  = cv.cvtColor(img, cv.COLOR_BGR2RGB)   # cara 1: cvtColor
img_rgb2 = img[:, :, ::-1]                      # cara 2: slicing kanal

print('piksel [256,256] BGR         :', img[256, 256])
print('piksel [256,256] RGB cvtColor:', img_rgb[256, 256])
print('piksel [256,256] RGB slicing :', img_rgb2[256, 256])
print('cvtColor == slicing          :', np.array_equal(img_rgb, img_rgb2))

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img);            ax[0].set_title('plt.imshow(img) — BGR dibaca sbg RGB (salah)')
ax[1].imshow(img_rgb);        ax[1].set_title('cv.cvtColor(BGR2RGB) (benar)')
ax[2].imshow(img_rgb2);       ax[2].set_title('img[:, :, ::-1] (benar)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## D2.2 — Citra keabuan (grayscale) dan colormap

Array 2D tidak menyimpan informasi warna; warna tampilan ditentukan **colormap**.
Colormap bawaan Matplotlib adalah `viridis` (ungu–kuning), bukan hitam–putih. Untuk
citra keabuan gunakan `cmap='gray'`, dan `cmap='magma'` sebagai contoh pewarnaan semu
(*pseudocolor*). Colormap hanya mengubah tampilan, bukan nilai array.

In [ ]:
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
print('gray.shape =', gray.shape, '| dtype =', gray.dtype,
      '| min =', int(gray.min()), '| max =', int(gray.max()))

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(gray);              ax[0].set_title("default 'viridis'")
ax[1].imshow(gray, cmap='gray'); ax[1].set_title("cmap='gray'")
ax[2].imshow(gray, cmap='magma');ax[2].set_title("cmap='magma' (pseudocolor)")
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

# bukti colormap tidak mengubah data: nilai array identik sebelum & sesudah ditampilkan
print('nilai gray tidak berubah:', gray.shape, int(gray.min()), int(gray.max()))

## D2.3 — Pengubahan ukuran (*resizing*) dan interpolasi

`cv.resize()` menerima ukuran dalam urutan **(lebar, tinggi)** — kebalikan dari `shape`.
Metode interpolasi memengaruhi hasil: `INTER_NEAREST` (tercepat, tepi bergerigi),
`INTER_LINEAR` (bawaan), `INTER_CUBIC` (lebih tajam untuk pembesaran), dan
`INTER_AREA` (paling baik untuk pengecilan karena menekan aliasing).

Berikut perbandingan ketiga metode **pembesaran** pada satu potongan (*crop*) yang
sama, agar perbedaan piksel terlihat jelas.

In [ ]:
h, w = img.shape[:2]
roi = img[h//4:h//2, w//4:w//2]          # potongan detail
rh, rw = roi.shape[:2]
UP = 4

methods = [('INTER_NEAREST', cv.INTER_NEAREST),
           ('INTER_LINEAR',  cv.INTER_LINEAR),
           ('INTER_CUBIC',   cv.INTER_CUBIC)]

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(cv.cvtColor(roi, cv.COLOR_BGR2RGB)); ax[0].set_title(f'asli {rw}x{rh}')
for i, (name, flag) in enumerate(methods, start=1):
    up = cv.resize(roi, (rw*UP, rh*UP), interpolation=flag)
    print(f'{name:14s} -> {up.shape[1]}x{up.shape[0]}')
    ax[i].imshow(cv.cvtColor(up, cv.COLOR_BGR2RGB)); ax[i].set_title(f'{name} (x{UP})')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# INTER_AREA untuk pengecilan, lalu dibesarkan kembali (bandingkan dgn asli)
small = cv.resize(img, (w//4, h//4), interpolation=cv.INTER_AREA)
big   = cv.resize(small, (w, h), interpolation=cv.INTER_CUBIC)
print('asli  :', img.shape)
print('small :', small.shape, '| nbytes:', small.nbytes)
print('big   :', big.shape)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB));   ax[0].set_title(f'asli {w}x{h}')
ax[1].imshow(cv.cvtColor(small, cv.COLOR_BGR2RGB)); ax[1].set_title(f'INTER_AREA {small.shape[1]}x{small.shape[0]}')
ax[2].imshow(cv.cvtColor(big, cv.COLOR_BGR2RGB));   ax[2].set_title(f'dibesarkan lagi {w}x{h}')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## D2.4 — Pencerminan (*flipping*)

`cv.flip(img, flipCode)`: `0` = vertikal (atas–bawah), `1` = horizontal (kiri–kanan),
`-1` = keduanya sekaligus. Operasi ini hanya menyusun ulang posisi piksel, tidak
mengubah nilainya, sehingga tidak ada informasi yang hilang.

In [ ]:
flip_v  = cv.flip(img, 0)    # vertikal
flip_h  = cv.flip(img, 1)    # horizontal
flip_vh = cv.flip(img, -1)   # keduanya

# bukti nilai piksel hanya dipindah, tidak berubah
print('sudut kiri-atas asli   :', img[0, 0])
print('sudut kiri-atas flip 0 :', flip_v[0, 0], '(= asli baris terbawah)')
print('total nilai sama       :', np.array_equal(np.sort(img.ravel()), np.sort(flip_v.ravel())))

fig, ax = plt.subplots(1, 4, figsize=(18, 5))
for a, (im, t) in zip(ax, [(img, 'asli'), (flip_v, 'flip 0 (vertikal)'),
                           (flip_h, 'flip 1 (horizontal)'), (flip_vh, 'flip -1 (keduanya)')]):
    a.imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB)); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

## D2.5 — Menggambar geometri & teks pada kanvas `int16`

Kanvas hitam dibuat dengan `np.zeros((tinggi, lebar, 3), dtype=np.int16)`. Seluruh fungsi
menggambar OpenCV menerima koordinat `(x, y)` dan **warna mengikuti urutan kanal citra**:
pada kanvas BGR, `(0, 0, 255)` berarti **merah**. `thickness=-1` mengisi penuh bentuk.

> **Catatan kompatibilitas:** `cv.putText()` di OpenCV 5.0 mensyaratkan kanvas bertipe
> `uint8` (`Assertion failed: img.depth() == CV_8U`). Karena itu, teks pada sel di bawah
> digambar di atas salinan `uint8` dari kanvas `int16`; bentuk geometri tetap digambar
> langsung di kanvas `int16`.

In [ ]:
H, W = 400, 600
canvas16 = np.zeros((H, W, 3), dtype=np.int16)
print('kanvas int16 ->', canvas16.shape, canvas16.dtype)

# 1) persegi panjang (garis tepi)
cv.rectangle(canvas16, (50, 50), (200, 150), (0, 255, 0), 3)

# 2) persegi panjang terisi penuh (thickness=-1)
cv.rectangle(canvas16, (250, 50), (400, 150), (255, 0, 0), -1)

# 3) lingkaran (garis tepi)
cv.circle(canvas16, (150, 300), 60, (0, 165, 255), 3)

# 4) lingkaran terisi + garis
cv.circle(canvas16, (400, 300), 60, (0, 0, 255), -1)
cv.line(canvas16, (50, 380), (550, 220), (255, 255, 255), 2)

show_any(canvas16, 'Kanvas int16: rectangle, rectangle terisi, circle, circle terisi, line')

# 5) teks: coba langsung pada int16 (gagal) lalu pada salinan uint8
try:
    cv.putText(canvas16, 'PCVK', (50, 30), cv.FONT_HERSHEY_SIMPLEX,
               1.0, (255, 255, 255), 2, cv.LINE_AA)
    print('putText pada int16: BERHASIL')
except cv.error as e:
    print('putText pada int16 GAGAL (butuh CV_8U):')
    print('  ', str(e).splitlines()[0])

canvas_text = canvas16.astype(np.uint8)
cv.putText(canvas_text, 'PCVK 2026', (50, 40), cv.FONT_HERSHEY_SIMPLEX,
           1.0, (255, 255, 255), 2, cv.LINE_AA)
cv2_imshow(canvas_text)

## D2.6 — Kanvas `int32`, `reshape`, dan `polylines`

Kanvas kedua dibuat bertipe `int32`. Untuk `cv.polylines()`, larik titik **wajib bertipe
`int32`**, dibentuk ulang menjadi `(-1, 1, 2)` dengan `reshape`, dan dibungkus dalam
`list`. Contoh di bawah memperlihatkan proses `reshape` dari larik datar menjadi bentuk
yang diterima OpenCV.

In [ ]:
canvas32 = np.zeros((H, W, 3), dtype=np.int32)
print('kanvas int32 ->', canvas32.shape, canvas32.dtype, '| nbytes:', canvas32.nbytes)

# reshape: larik datar -> (-1, 1, 2)
flat = np.array([300, 60, 480, 200, 360, 340, 240, 200], dtype=np.int32)
print('flat.shape   :', flat.shape)
pts = flat.reshape((-1, 1, 2))
print('setelah reshape:', pts.shape, pts.dtype)

cv.polylines(canvas32, [pts], isClosed=True, color=(0, 0, 255), thickness=3)

# tambahan: segitiga terbuka + kotak untuk melengkapi
tri = np.array([[60, 340], [180, 340], [120, 200]], dtype=np.int32).reshape((-1, 1, 2))
cv.polylines(canvas32, [tri], isClosed=False, color=(255, 255, 0), thickness=2)
cv.rectangle(canvas32, (400, 250), (550, 370), (0, 255, 0), 2)

print('nilai piksel kanvas32 min/max:', int(canvas32.min()), int(canvas32.max()))
show_any(canvas32, 'Kanvas int32: polylines (reshape) + segitiga + rectangle')

## D2.7 — Menyimpan hasil

Hasil disimpan ke folder kerja (`/kaggle/working` di Kaggle, folder lokal bila dijalankan
di komputer). `cv.imwrite()` menerima kanvas `uint8` maupun `int16/int32` (akan dikonversi
ke 8-bit saat penulisan berkas).

In [ ]:
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
f_out = os.path.join(OUT, 'week2_canvas_int32.png')
cv.imwrite(f_out, canvas32)
print('disimpan:', f_out, '| ada?', os.path.exists(f_out))

f_out2 = os.path.join(OUT, 'week2_gray.png')
cv.imwrite(f_out2, gray)
print('disimpan:', f_out2, '| ada?', os.path.exists(f_out2))

---

# PERTANYAAN PRAKTIKUM

Setiap jawaban disertai keluaran program.

## Pertanyaan 1 — Mengapa hasil `cv2_imshow()` dan `plt.imshow()` berbeda (tanpa konversi)?

Tampilkan satu citra bebas dengan `cv2_imshow()` dan `plt.imshow()` tanpa konversi warna,
lalu cetak nilai satu piksel yang sama sebagai bukti.

In [ ]:
free = cv.imread(paths['potret.jpg'])

# pilih piksel dengan selisih kanal B dan R terbesar agar buktinya paling jelas
_b = free[:, :, 0].astype(int)
_r = free[:, :, 2].astype(int)
yy, xx = np.unravel_index(np.argmax(np.abs(_b - _r)), _b.shape)
print(f'piksel terpilih (y,x) = ({yy}, {xx})')
print('nilai larik (urutan penyimpanan) :', free[yy, xx])
print('B =', free[yy, xx, 0], '| G =', free[yy, xx, 1], '| R =', free[yy, xx, 2])

cv2_imshow(free)          # cv2_imshow: indeks 0 diperlakukan sebagai B -> benar

plt.figure(figsize=(5, 5))
plt.imshow(free)          # plt.imshow: indeks 0 diperlakukan sebagai R -> tertukar
plt.title('plt.imshow(free) tanpa konversi')
plt.axis('off')
plt.show()

### Jawaban 1

Kedua fungsi menerima **larik yang sama persis** dari `cv.imread()`, tetapi berbeda dalam
**menafsirkan urutan kanal**:

* `cv.imread()` menyimpan piksel dalam urutan **BGR** (indeks 0 = Biru, 1 = Hijau, 2 = Merah).
  `cv2_imshow()` mengasumsikan masukan BGR, sehingga warna yang tampil **benar**.
* `plt.imshow()` mengasumsikan masukan **RGB** (indeks 0 = Merah). Karena indeks 0 pada
  larik kita sebenarnya Biru, Matplotlib menampilkannya sebagai Merah — kanal merah dan
  biru **tertukar** (kulit/latar tampak kebiruan).

Bukti dari keluaran program: piksel terpilih `(y, x) = (382, 180)` bernilai
`[B, G, R] = [19, 76, 151]`. `cv2_imshow()` membaca `19` sebagai **Biru** dan `151` sebagai
**Merah** → warna benar. `plt.imshow()` membaca `19` sebagai **Merah** dan `151` sebagai
**Biru** → merah dan biru bertukar. Jadi perbedaan tampilan bukan karena datanya berbeda,
melainkan karena **asumsi urutan kanal** kedua fungsi berbeda; solusinya memanggil
`cv.cvtColor(img, cv.COLOR_BGR2RGB)` sebelum `plt.imshow()`.

## Pertanyaan 2 — `int16` vs `int32`: dtype, `min()`, `max()`, `nbytes`, dan memori

In [ ]:
c16 = np.zeros((H, W, 3), dtype=np.int16)
c32 = np.zeros((H, W, 3), dtype=np.int32)

# gambar bentuk identik pada keduanya
for c in (c16, c32):
    cv.rectangle(c, (50, 50), (200, 150), (0, 255, 0), 3)
    cv.circle(c, (400, 300), 60, (255, 0, 0), -1)

for name, c in [('int16', c16), ('int32', c32)]:
    print(f'{name}: dtype={c.dtype}  min={c.min()}  max={c.max()}  '
          f'nbytes={c.nbytes}  itemsize={c.itemsize} byte')

print('\nrasio nbytes int32/int16 =', c32.nbytes / c16.nbytes)
print('piksel (60,60) int16:', c16[60, 60], '| int32:', c32[60, 60])
print('semua nilai sama?', np.array_equal(c16.astype(np.int64), c32.astype(np.int64)))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (c, t) in zip(ax, [(c16, 'int16'), (c32, 'int32')]):
    a.imshow(cv.cvtColor(c.astype(np.uint8), cv.COLOR_BGR2RGB))
    a.set_title(f'kanvas {t}')
    a.axis('off')
plt.tight_layout(); plt.show()

### Jawaban 2

| Properti | `int16` | `int32` |
|---|---|---|
| `dtype` | `int16` | `int32` |
| Jangkauan nilai | −32.768 … 32.767 | −2.147.483.648 … 2.147.483.647 |
| `itemsize` | 2 byte | 4 byte |
| `nbytes` kanvas 400×600×3 | **1.440.000** byte | **2.880.000** byte |
| `min()` / `max()` (setelah digambar) | 0 / 255 | 0 / 255 |

* **Tampilan tidak berbeda.** Kedua kanvas digambar dengan bentuk, warna, dan koordinat
  identik; karena nilai pikselnya sama (`min=0`, `max=255`, dan seluruh elemen terbukti
  sama), hasil visualnya pun identik.
* **Ukuran memori berbeda 2×.** `int32` memakai 4 byte per elemen sedangkan `int16` hanya
  2 byte, sehingga untuk 400×600×3 = 720.000 elemen diperoleh `nbytes` `int32` =
  2.880.000 byte dan `int16` = 1.440.000 byte (rasio 2,0 — terbukti dari keluaran program).
  Keduanya jauh lebih besar daripada kanvas `uint8` (720.000 byte) yang lazim untuk citra
  8-bit.
* **Alasan:** `nbytes = jumlah elemen × itemsize`. Nilai 0–255 masih muat di kedua tipe,
  jadi isi dan tampilan sama, tetapi lebar penyimpanan tiap elemen berbeda (16 bit vs
  32 bit). `int16` dipakai untuk menampung nilai negatif (mis. hasil selisih dua citra),
  sedangkan `int32` lebih jarang untuk piksel namun **wajib** untuk larik titik
  `cv.polylines()`.

## Pertanyaan 3 — Kegunaan `from google.colab.patches import cv2_imshow` dan mengapa `cv.imshow()` tidak bisa dipakai di Colab

In [ ]:
import importlib.util
print('modul google.colab tersedia?', importlib.util.find_spec('google.colab') is not None)
print('cv2_imshow yang dipakai sekarang ->', getattr(cv2_imshow, '__module__', 'shim lokal'))
print('cv.imshow tersedia di cv2?', hasattr(cv, 'imshow'))

# tampilkan baris GUI pada build information OpenCV
info = cv.getBuildInformation()
gui = [ln.strip() for ln in info.splitlines()
       if 'GUI' in ln or 'Display' in ln or 'GTK' in ln or 'Win32' in ln]
print('backend GUI terdeteksi:', gui if gui else '(tidak ada backend GUI)')

### Jawaban 3

`from google.colab.patches import cv2_imshow` mengimpor **`cv2_imshow()`**, fungsi pengganti
yang disediakan Colab untuk menampilkan citra sebagai **keluaran sel** notebook. Masukannya
dianggap berurutan **BGR** (sama seperti `cv.imread()`), lalu citra dirender ke output sel.

`cv.imshow()` bawaan OpenCV **tidak dapat dipakai** di Google Colab (dan juga Kaggle) karena
fungsi itu membutuhkan **jendela GUI / display server** sistem operasi (X11/Wayland/Win32).
Server Colab/Kaggle berjalan **headless** — tidak ada layar, jendela, maupun event loop GUI.
`cv.imshow()` juga menuntut pasangan `cv.waitKey()` dan `cv.destroyAllWindows()` yang justru
memblokir proses. Karena itu di lingkungan notebook dipakai `cv2_imshow()` (Colab) atau
`plt.imshow()` (Matplotlib menggambar ke buffer gambar, aman tanpa GUI).

Pada notebook ini, sel header menyiapkan **shim** yang sama: bila `google.colab` tidak ada
(kasus Kaggle/lokal), didefinisikan `cv2_imshow(img)` yang memanggil `plt.imshow()`, sehingga
kode praktikum tetap jalan tanpa perubahan. Keluaran program di atas menunjukkan
`modul google.colab tersedia? False` di Kaggle — itulah alasan shim diperlukan.

## Pertanyaan 4 — `skimage.io.imread()` vs `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`: mana yang benar?

`skimage.io.imread()` membaca citra dalam urutan **RGB**, berbeda dengan `cv2.imread()`
yang **BGR**. Bukti di bawah membandingkan nilai piksel, bukan sekadar pengamatan visual.

In [ ]:
from skimage import io as skio

sk    = skio.imread(paths['potret.jpg'])         # RGB (sudah benar)
cvim  = cv.imread(paths['potret.jpg'])           # BGR
conv  = cv.cvtColor(cvim, cv.COLOR_BGR2RGB)      # konversi BGR->RGB (benar)
salah = cv.cvtColor(sk, cv.COLOR_BGR2RGB)        # konversi atas citra yg SUDAH RGB (salah)

yy, xx = np.unravel_index(np.argmax(np.abs(cvim[:, :, 0].astype(int) -
                                           cvim[:, :, 2].astype(int))), cvim.shape[:2])
print(f'piksel acuan (y,x) = ({yy}, {xx})')
print('skimage.io.imread (RGB)     :', sk[yy, xx])
print('cv2.imread (BGR)            :', cvim[yy, xx])
print('cvtColor(cvim, BGR2RGB)     :', conv[yy, xx])
print('cvtColor(sk, BGR2RGB)       :', salah[yy, xx], '  <- kanal tertukar')
print('sk == cvtColor(cvim)?       :', np.array_equal(sk, conv))
print('cvtColor(sk) == sk?         :', np.array_equal(salah, sk))

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(sk);    ax[0].set_title('skimage.io.imread() — RGB (BENAR)')
ax[1].imshow(conv);  ax[1].set_title('cv2.cvtColor(cvim, BGR2RGB) — BENAR')
ax[2].imshow(salah); ax[2].set_title('cv2.cvtColor(sk, BGR2RGB) — kanal TERTUKAR')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

### Jawaban 4

* `skimage.io.imread()` mengembalikan larik berurutan **RGB**, jadi menampilkannya langsung
  dengan `plt.imshow()` menghasilkan **warna yang sebenarnya**.
* `cv.cvtColor(img, cv.COLOR_BGR2RGB)` **hanya benar bila `img` berasal dari `cv2.imread()`
  (BGR)**. Diterapkan pada citra yang sudah RGB (hasil skimage), konversi ini justru
  **menukar kanal merah dan biru** sehingga salah.

Bukti keluaran program pada piksel acuan `(y, x) = (382, 180)`:
`skimage = [151, 76, 19]`, sedangkan `cv2.imread = [19, 76, 151]` — terlihat `skimage`
adalah kebalikan urutan dari `cv2.imread`. `cvtColor(cvim, BGR2RGB) = [151, 76, 19]`
**sama dengan** `skimage` (`sk == cvtColor(cvim)? True`), membuktikan hasil konversi itu RGB
yang benar. Sebaliknya `cvtColor(sk, BGR2RGB) = [19, 76, 151]` adalah kebalikannya — kanal
tertukar, dan `cvtColor(sk) == sk? False`.

**Kesimpulan:** citra `skimage` menampilkan warna sebenarnya; `cvtColor(BGR2RGB)` pada citra
yang sama justru menukar kanal. Aturan praktisnya: konversi `BGR2RGB` hanya dikenakan **sekali**
pada citra keluaran OpenCV sebelum ditampilkan dengan Matplotlib.

## Pertanyaan 5 — Apakah `figsize` yang besar mengubah `img.shape`?

In [ ]:
print('shape SEBELUM ditampilkan :', img.shape)

show(img, 'figsize=(2,2) — tampilan kecil', figsize=(2, 2))
show(img, 'figsize=(14,14) — tampilan besar', figsize=(14, 14))

print('shape SESUDAH ditampilkan :', img.shape)
print('nbytes citra tetap        :', img.nbytes)

### Jawaban 5

**Tidak.** `img.shape` tetap `(512, 512, 3)` baik sebelum maupun sesudah ditampilkan dengan
`figsize=(2,2)` maupun `figsize=(14,14)` (terbukti dari keluaran program: shape sama dan
`nbytes` tetap).

* `figsize` pada `plt.figure()` hanya mengatur **ukuran kanvas gambar (figure) dalam inci**
  saat *penampilan*. Ini sekadar penskalaan tampilan yang dilakukan backend
  Matplotlib/`imshow`, **tidak menyentuh** larik NumPy.
* `cv2.resize()` benar-benar **mengubah jumlah piksel** citra sehingga `img.shape` berubah.
  Interpolasi menghitung nilai piksel baru dan **tidak menciptakan informasi baru** — citra
  yang diperbesar tidak mengembalikan detail yang hilang saat sampling.

Jadi mengubah ukuran **tampilan** (figsize) dan mengubah ukuran **citra** (`cv2.resize`)
adalah dua hal berbeda: hanya yang kedua mengubah data.

---

# TUGAS PRAKTIKUM — Penerapan

## Tugas 1 — Kombinasi kanal Merah-Biru dan Hijau-Biru (kanal lain dinolkan)

Karena `cv.imread()` berurutan **BGR**, maka: kanal R = indeks 2, G = indeks 1, B = indeks 0.
Kombinasi **Merah-Biru** dibuat dengan menolkan kanal G (indeks 1); kombinasi
**Hijau-Biru** dibuat dengan menolkan kanal R (indeks 2). Keduanya disusun berdampingan
dalam satu figure berjudul.

In [ ]:
rb = img.copy(); rb[:, :, 1] = 0    # nol-kan Green  -> hanya Red + Blue
gb = img.copy(); gb[:, :, 2] = 0    # nol-kan Red    -> hanya Green + Blue

print('kanal RB: nilai G rata-rata =', rb[:, :, 1].mean(), '(harus 0)')
print('kanal GB: nilai R rata-rata =', gb[:, :, 2].mean(), '(harus 0)')

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB)); ax[0].set_title('Asli (BGR->RGB)')
ax[1].imshow(cv.cvtColor(rb, cv.COLOR_BGR2RGB));  ax[1].set_title('Kombinasi Merah-Biru (G=0)')
ax[2].imshow(cv.cvtColor(gb, cv.COLOR_BGR2RGB));  ax[2].set_title('Kombinasi Hijau-Biru (R=0)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## Tugas 2 — Tiga bentuk pencerminan dalam satu figure

`cv.flip()` dengan `flipCode` 0 (vertikal), 1 (horizontal), dan −1 (keduanya) ditampilkan
sebagai tiga subplot berlabel.

In [ ]:
flips = [('Vertikal (flipCode=0)', cv.flip(img, 0)),
         ('Horizontal (flipCode=1)', cv.flip(img, 1)),
         ('Vertikal+Horizontal (flipCode=-1)', cv.flip(img, -1))]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (t, f) in zip(ax, flips):
    a.imshow(cv.cvtColor(f, cv.COLOR_BGR2RGB)); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

# verifikasi: flip ganda = rotasi 180 derajat
print('flip(-1) == rotasi 180?', np.array_equal(cv.flip(img, -1), cv.rotate(img, cv.ROTATE_180)))

## Tugas 3 — Rectangle, circle, dan teks nama pada foto aktivitas

Objek gambar ditentukan **secara proporsional** terhadap ukuran citra (`h`, `w`), lalu
ditambah teks nama dengan font, ukuran, dan warna pilihan sendiri. Foto yang dipakai adalah
`foto_wajah.jpg` (foto aktivitas yang menampilkan wajah, bukan pasfoto).

In [ ]:
NAME = 'Singgih Wahyu Permana'   # ganti sesuai nama Anda

drawn = img.copy()
h, w = drawn.shape[:2]

# kotak wajah (proporsional: bagian tengah-atas citra)
x1, y1 = int(w * 0.28), int(h * 0.12)
x2, y2 = int(w * 0.72), int(h * 0.66)
cv.rectangle(drawn, (x1, y1), (x2, y2), (0, 255, 255), max(1, h // 150))

# lingkaran penanda di pusat kotak (proporsional)
cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
r = int(min(x2 - x1, y2 - y1) * 0.45)
cv.circle(drawn, (cx, cy), r, (0, 0, 255), max(1, h // 150))

# teks nama: font, ukuran, dan warna pilihan sendiri
font = cv.FONT_HERSHEY_SCRIPT_SIMPLEX
fscale = h / 420.0
fthick = max(1, h // 220)
(tw, th), _ = cv.getTextSize(NAME, font, fscale, fthick)
tx, ty = int(w * 0.03), h - int(h * 0.03)
cv.putText(drawn, NAME, (tx, ty), font, fscale, (0, 255, 0), fthick, cv.LINE_AA)
cv.line(drawn, (tx, ty + max(2, fthick)), (tx + tw, ty + max(2, fthick)), (0, 255, 0), fthick)

print('ukuran citra:', (w, h), '| font:', font, '| fontScale: %.2f' % fscale,
      '| thickness:', fthick, '| warna teks: (0,255,0) = hijau (BGR)')
cv2_imshow(drawn)

---

# TUGAS PRAKTIKUM — Analisa

## Pertanyaan yang dirumuskan

> **Berapa persen piksel pada `foto_wajah.jpg` yang bernilai keabuan lebih gelap dari 100,
> dan apakah mengecilkan lalu membesarkan kembali citra mengubah rata-rata intensitasnya?**

## Rancangan program

1. Ubah `foto_wajah.jpg` menjadi keabuan (`cv.cvtColor(..., COLOR_BGR2GRAY)`).
2. Hitung persentase piksel dengan nilai `< 100` (piksel gelap) memakai perbandingan larik
   NumPy dan `mean()`.
3. Kecilkan citra keabuan dengan `cv2.resize(..., INTER_AREA)`, lalu besarkan kembali dengan
   `cv2.resize(..., INTER_CUBIC)`.
4. Bandingkan rata-rata intensitas citra asli, kecil, dan hasil pembesaran.

In [ ]:
g = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
h, w = g.shape

# 1) persentase piksel gelap (<100)
gelap = int((g < 100).sum())
persen = gelap / g.size * 100
print(f'jumlah piksel       : {g.size}')
print(f'piksel < 100        : {gelap}  ({persen:.2f}%)')
print(f'rata-rata keabuan   : {g.mean():.4f}')

# 2) kecilkan lalu besarkan kembali
g_small = cv.resize(g, (w // 4, h // 4), interpolation=cv.INTER_AREA)
g_back  = cv.resize(g_small, (w, h), interpolation=cv.INTER_CUBIC)
print(f'rata-rata asli      : {g.mean():.4f}')
print(f'rata-rata kecil     : {g_small.mean():.4f}')
print(f'rata-rata dibesarkan: {g_back.mean():.4f}')
print(f'selisih asli vs dibesarkan: {abs(g.mean() - g_back.mean()):.4f}')

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(g, cmap='gray');       ax[0].set_title(f'asli {w}x{h}')
ax[1].imshow(g_small, cmap='gray'); ax[1].set_title(f'kecil {g_small.shape[1]}x{g_small.shape[0]}')
ax[2].imshow(g_back, cmap='gray');  ax[2].set_title('dibesarkan kembali')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## Data hasil dan kesimpulan

Keluaran program:

* Jumlah piksel citra `foto_wajah.jpg` (512×512) = **262.144**.
* Piksel dengan intensitas keabuan **< 100** = **102.065 piksel (≈ 38,93 %)**. Artinya
  lebih dari sepertiga citra tergolong gelap (area rambut, bayangan, dan latar gelap).
* Rata-rata intensitas: asli ≈ **115,4189**; setelah dikecilkan (128×128) ≈ **115,4158**;
  setelah dibesarkan kembali (512×512) ≈ **115,4459**. Selisih asli vs hasil pembesaran
  hanya ≈ **0,027** satuan intensitas (≈ 0,02 %).

**Kesimpulan:** (1) sekitar 39 % piksel foto lebih gelap dari nilai 100, sehingga citra ini
cenderung agak gelap (rata-rata keabuan ≈ 115 dari maksimum 255); (2) **mengecilkan lalu
membesarkan citra hampir tidak mengubah rata-rata intensitasnya** — selisihnya jauh di bawah
1 satuan. Ini masuk akal karena `INTER_AREA` pada pengecilan melakukan perata-rataan area
(menjaga rata-rata lokal), dan pembesaran tidak menciptakan informasi baru. Yang hilang
akibat *downscaling* bukan rata-rata, melainkan **detail/ketajaman** — itulah sebabnya citra
yang dikecilkan lalu dibesarkan tampak lebih kabur meski rata-ratanya hampir sama.

---

# TUGAS PRAKTIKUM — Aplikasi

Batasan: hanya memakai fungsi yang dipelajari pada modul ini (pembacaan berkas, pengirisan,
`cv2.resize`, `cv2.flip`, `cv2.cvtColor`, fungsi penggambaran OpenCV, dan kanvas NumPy).
**Semua ukuran, posisi, dan ketebalan dihitung dari ukuran citra masukan** (adaptif) — tidak
ada angka piksel tetap pada kode.

## Studi Kasus 1 — Bar label identitas adaptif

Panitia perlu membubuhkan label identitas pada ratusan foto dokumentasi berukuran
bermacam-macam. Bar dan ukuran huruf harus menyesuaikan tinggi citra sehingga proporsinya
tetap seimbang pada semua foto.

Fungsi `add_label()` menghitung tinggi bar sebagai **persentase tinggi citra**, lalu menurunkan
skala huruf dan ketebalan dari tinggi bar tersebut. Bila teks ternyata lebih lebar dari ruang
tersedia, skala huruf diperkecil otomatis (loop adaptif). Fungsi yang sama dikenakan pada dua
citra berbeda ukuran: `foto_wajah.jpg` (512×512) dan `lanskap.jpg` (405×720).

In [ ]:
def add_label(img, name, bar_frac=0.10):
    """Bubuhkan bar label di bawah citra; semua ukuran dihitung dari ukuran citra."""
    out = img.copy()
    h, w = out.shape[:2]

    # tinggi bar & margin proporsional terhadap tinggi citra
    bar_h = max(1, int(round(h * bar_frac)))
    margin = max(1, int(round(w * 0.03)))
    thick  = max(1, int(round(bar_h / 18)))

    # bar gelap menutupi bagian bawah
    cv.rectangle(out, (0, h - bar_h), (w, h), (0, 0, 0), cv.FILLED)

    # skala huruf dari tinggi bar; kecilkan otomatis bila teks terlalu lebar
    font = cv.FONT_HERSHEY_DUPLEX
    fscale = bar_h / 45.0
    (tw, th), _ = cv.getTextSize(name, font, fscale, thick)
    while tw > (w - 2 * margin) and fscale > 0.1:
        fscale *= 0.9
        (tw, th), _ = cv.getTextSize(name, font, fscale, thick)

    x = margin
    y = h - bar_h + (bar_h + th) // 2          # teks di tengah bar secara vertikal
    cv.putText(out, name, (x, y), font, fscale, (255, 255, 255), thick, cv.LINE_AA)

    # garis aksen di tepi atas bar, panjang mengikuti lebar teks
    cv.line(out, (x, h - bar_h), (x + tw, h - bar_h), (0, 200, 255), max(1, thick // 2))
    return out

NAME = 'Singgih Wahyu Permana'
tests = [('foto_wajah.jpg', imgs['foto_wajah.jpg']), ('lanskap.jpg', imgs['lanskap.jpg'])]

labeled = []
for fn, im in tests:
    res = add_label(im, NAME)
    h, w = im.shape[:2]
    print(f'{fn:16s} input {w}x{h} -> bar {max(1, int(round(h*0.10)))} px '
          f'({10}% tinggi), margin {max(1, int(round(w*0.03)))} px')
    labeled.append((fn, res))

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, (fn, res) in zip(ax, labeled):
    a.imshow(cv.cvtColor(res, cv.COLOR_BGR2RGB)); a.set_title(f'label pada {fn}'); a.axis('off')
plt.tight_layout(); plt.show()

### Analisis Studi Kasus 1

Semua parameter diturunkan dari ukuran citra, sehingga **proporsi label tetap sama** pada
kedua foto yang ukurannya berbeda:

| Citra | Ukuran (w×h) | Tinggi bar (10 % h) | Ketebalan huruf | Skala huruf |
|---|---|---|---|---|
| `foto_wajah.jpg` | 512×512 | 51 px | 2 px | 51/45 ≈ 1,13 |
| `lanskap.jpg` | 720×405 | 41 px | 2 px | 41/45 ≈ 0,91 |

Karena bar dan huruf dihitung dari tinggi citra (bukan angka tetap), label tampak **seimbang
di kedua foto** — bar selalu ±10 % tinggi citra dan teks selalu muat di dalam bar (loop
penyesuaian skala menjamin `tw ≤ w − 2·margin`). Menambah foto lain cukup memanggil
`add_label()` tanpa penyetelan ulang, sesuai kebutuhan panitia.

## Studi Kasus 2 — Thumbnail 1:1 dengan *letterbox* dan logo toko

Marketplace menampilkan foto produk dalam bingkai **persegi 1:1**. Foto potret/lanskap yang
diunggah apa adanya akan terpotong atau terlihat tidak proporsional. Program berikut mengubah
foto berukuran apa pun menjadi thumbnail persegi **tanpa mengubah rasio aslinya**, mengisi
sisa ruang dengan kanvas berwarna (*letterbox*), menempatkan citra **tepat di tengah**, dan
menambahkan **logo nama toko proporsional** di bagian bawah.

Seluruh ukuran (sisi kanvas, ukuran citra, posisi tengah, tinggi strip logo, skala huruf)
dihitung dari ukuran citra masukan.

In [ ]:
def make_thumbnail(img, store='TOKO PCVK', bg=(235, 235, 235), strip=(45, 45, 45)):
    """Ubah citra apa pun menjadi thumbnail persegi 1:1 (letterbox) + logo toko."""
    h, w = img.shape[:2]
    side = max(h, w)                                   # sisi kanvas = dimensi terpanjang
    canvas = np.full((side, side, 3), bg, dtype=np.uint8)

    # skala rasio tetap: dimensi terpanjang = sisi, dimensi lain menyusut proporsional
    scale = side / max(h, w)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    resized = cv.resize(img, (new_w, new_h), interpolation=cv.INTER_AREA)

    # tempel di tengah kanvas (letterbox)
    x0, y0 = (side - new_w) // 2, (side - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized

    # strip logo proporsional di dasar kanvas
    strip_h = max(1, int(round(side * 0.10)))
    cv.rectangle(canvas, (0, side - strip_h), (side, side), strip, cv.FILLED)

    font = cv.FONT_HERSHEY_DUPLEX
    fscale = strip_h / 45.0
    thick = max(1, int(round(strip_h / 20)))
    (tw, th), _ = cv.getTextSize(store, font, fscale, thick)
    while tw > side - 2 * int(round(side * 0.04)) and fscale > 0.1:
        fscale *= 0.9
        (tw, th), _ = cv.getTextSize(store, font, fscale, thick)
    org = ((side - tw) // 2, side - strip_h + (strip_h + th) // 2)
    cv.putText(canvas, store, org, font, fscale, (255, 255, 255), thick, cv.LINE_AA)
    return canvas

for fn in ['potret.jpg', 'lanskap.jpg']:
    src = imgs[fn]
    th_img = make_thumbnail(src, store='TOKO PCVK')
    h, w = src.shape[:2]
    print(f'{fn:12s} input {w}x{h} -> thumbnail {th_img.shape[1]}x{th_img.shape[0]} '
          f'(1:1), letterbox {abs(th_img.shape[0]-th_img.shape[1])} px')
    show_rgb(cv.cvtColor(th_img, cv.COLOR_BGR2RGB),
             f'thumbnail 1:1 dari {fn} ({w}x{h})', figsize=(5, 5))

### Analisis Studi Kasus 2

* **Rasio asli terjaga.** Faktor skala `side / max(h, w)` membuat dimensi terpanjang sama
  dengan sisi kanvas dan dimensi lainnya menyusut dengan rasio yang sama, sehingga tidak ada
  peregangan maupun pemotongan.
* **Letterbox & posisi tengah.** Selisih sisi dengan citra yang sudah diskalakan diisi warna
  latar `bg`; offset `(side − new_w)//2` dan `(side − new_h)//2` menempatkan citra tepat di
  tengah. `potret.jpg` (360×540) menghasilkan padding di kiri–kanan, sedangkan `lanskap.jpg`
  (720×405) menghasilkan padding di atas–bawah.
* **Logo proporsional.** Tinggi strip, skala huruf, dan ketebalan dihitung dari `side`, jadi
  logo tampak sama besar relatif terhadap thumbnail apa pun. Karena tidak ada angka piksel
  tetap, fungsi ini langsung dapat dipakai untuk semua foto penjual tanpa penyetelan.

---

## Kesimpulan

Seluruh praktikum Modul 2 telah dijalankan di lingkungan Kaggle: pembacaan citra tanpa Google
Drive, penampilan via `cv2_imshow()`/`plt.imshow()`, konversi BGR↔RGB dan keabuan beserta
colormap, `resize` dengan empat metode interpolasi, `flip`, penggambaran geometri/teks pada
kanvas `int16` dan `int32`, `reshape`, `polylines`, dan penyimpanan berkas. Pertanyaan 1–5,
Tugas Penerapan 1–3, Tugas Analisa, serta Tugas Aplikasi (Studi Kasus 1 & 2) telah
diimplementasikan; kedua studi kasus aplikasi bersifat **adaptif** (semua ukuran/posisi
dihitung dari ukuran citra masukan).